# Port-a-Prof: QLoRA Fine-Tuning on gemma-4-E2B-it

## 1. Configuration

In [ ]:
from pathlib import Path

# ── Paths ── 
# TRAINING_DIR       : diretório com os arquivos JSON brutos por problema
# INTERMEDIATE_JSONL : todos os exemplos extraídos antes da divisão treino/validação
# TRAIN_JSONL        : divisão de treino de 80%
# VAL_JSONL          : divisão de validação de 20%
TRAINING_DIR       = Path("dataset")
INTERMEDIATE_JSONL = Path("training_data_intermediate.jsonl")
TRAIN_JSONL        = Path("port_a_prof_train.jsonl")
VAL_JSONL          = Path("port_a_prof_val.jsonl")

# ── Modelo base ── 
MODEL_ID = "google/gemma-4-E2B-it"

assert TRAINING_DIR.exists(), f"Training directory not found: {TRAINING_DIR.resolve()}"
print("Training dir:", TRAINING_DIR.resolve())


## 2. Converter arquivos JSON em exemplos de treinamento no formato de chat

Cada arquivo JSON bruto contém um problema de nível ensino médio e uma lista de *trajetórias*.
A trajectory is a sequence of tutoring-session snapshots — one per turn —
where each entry records:

- `dialogue_history` — a conversa até o momento (turnos do aluno + assistente)
- `internal_state`   — `current_status`, `teacher_role`, and optionally `support`
- `target_teacher_response` — a resposta correta de referência que o modelo deve produzir

A entrada usada para o ajuste fino é estruturada como:

```
## PROBLEMA
<texto do problema>

## TENTATIVA_DO_ALUNO
<mensagem mais recente do aluno>

## STATUS
<misconception / support need label>

## PAPEL_DO_PROFESSOR
<role label>

## SUPORTE          ← only present when TEACHER_ROLE == partial_worked_step
<low/medium/high>
```



In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List


def extract_latest_student_attempt(dialogue_history: List[Dict[str, str]]) -> str:
    """
    Retorna a mensagem mais recente do histórico de diálogo.
    """
    student_roles = {"student", "user"}
    for turn in reversed(dialogue_history):
        role    = (turn.get("role")    or "").strip().lower()
        content = (turn.get("content") or "").strip()
        if role in student_roles and content:
            return content
    return ""


def build_user_content(
    problem: str,
    student_attempt: str,
    internal_state: Dict[str, Any],
) -> str:
    """
    Monta o prompt estruturado do usuário a partir de suas partes.

    Os campos STATUS e PAPEL_DO_PROFESSOR codificam o contexto pedagógico que
    o modelo deve aprender a usar. O bloco SUPORTE é apenas
    incluído para o papel `partial_worked_step`.
    """
    base = (
        f"## PROBLEMA\n{problem.strip()}\n\n"
        f"## TENTATIVA_DO_ALUNO\n{student_attempt.strip()}\n\n"
        f"## STATUS\n{internal_state['current_status'].strip()}\n\n"
        f"## PAPEL_DO_PROFESSOR\n{internal_state['teacher_role'].strip()}"
    )
    # Adiciona o fragmento de suporte apenas quando o papel exigir
    if internal_state['teacher_role'].strip() == "partial_worked_step":
        base += f"\n\n## SUPORTE\n{internal_state['support'].strip()}"
    return base


def extract_examples_from_problem_file(file_path: Path) -> List[Dict[str, Any]]:
    """
    Interpreta um único arquivo JSON de problema e retorna uma lista de exemplos de treinamento
    in HuggingFace chat format:

        {"messages": [{"role": "user", "content": ...},
                      {"role": "assistant", "content": ...}]}
    """
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    problem     = data["problem"]
    trajectories = data["trajectories"]

    rows: List[Dict[str, Any]] = []
    for trajectory in trajectories:
        for entry in trajectory["entries"]:
            dialogue_history = entry["dialogue_history"]
            internal_state   = entry["internal_state"]
            target           = entry["target_teacher_response"].strip()

            student_attempt = extract_latest_student_attempt(dialogue_history)
            if not student_attempt:
                continue  # pula se nenhum turno do aluno for encontrado

            user_content = build_user_content(
                problem=problem,
                student_attempt=student_attempt,
                internal_state=internal_state,
            )

            rows.append({
                "messages": [
                    {"role": "user",      "content": user_content},
                    {"role": "assistant", "content": target},
                ]
            })
    return rows


# Itera por cada JSON de problema e coleta todas as linhas de treinamento 
all_rows: List[Dict[str, Any]] = []

for json_path in sorted(TRAINING_DIR.rglob("*.json")):
    all_rows.extend(extract_examples_from_problem_file(json_path))

print(f"Extracted {len(all_rows)} total training examples.")

# Salva o conjunto intermediário completo antes da divisão (útil para inspeção)
with INTERMEDIATE_JSONL.open("w", encoding="utf-8") as f:
    for row in all_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote: {INTERMEDIATE_JSONL.resolve()}")


## 3. Quick sanity check

In [ ]:
import random

# Prints a random example 
sample = random.choice(all_rows)
print(json.dumps(sample, indent=2, ensure_ascii=False)[:4000])


## 4. Train / validation split


In [ ]:
from sklearn.model_selection import train_test_split

# Divisão 80/20 — random_state=42 garante a mesma divisão a cada execução
train_rows, val_rows = train_test_split(
    all_rows,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"Train rows: {len(train_rows)}")
print(f"Val rows:   {len(val_rows)}")

# Write both splits to disk in JSONL format (one JSON object per line)
with TRAIN_JSONL.open("w", encoding="utf-8") as f:
    for row in train_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with VAL_JSONL.open("w", encoding="utf-8") as f:
    for row in val_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote: {TRAIN_JSONL.resolve()}")
print(f"Wrote: {VAL_JSONL.resolve()}")


## 5. Carregar dataset

Carrega os splits JSONL em um `DatasetDict` do HuggingFace para ficarem compatíveis
com o `SFTTrainer` nas etapas seguintes.


In [ ]:
from datasets import load_dataset

# Each JSONL line is a {"messages": [...]} object — 'json' format handles this
dataset = load_dataset(
    "json",
    data_files={
        "train":      str(TRAIN_JSONL),
        "validation": str(VAL_JSONL),
    },
)

dataset


## 6. Apply chat template

Gemma instruction-tuned models expect a specific conversation format with
special tokens that distinguish user and assistant turns.  

`enable_thinking=False` desativa os tokens de raciocínio do Gemma 4 para que o
modelo gere respostas diretamente.


In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")

def formatting_prompts_func(examples):
    """
    Apply Gemma's instruction chat template to each conversation.

    tokenize=False        → return a plain string, not token IDs
    add_generation_prompt → False porque a resposta-alvo está incluída
    enable_thinking       → False to suppress chain-of-thought tokens
    """
    texts = [
            processor.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        for convo in examples["messages"]
    ]
    return {"text": texts}

# Aplica o template a cada split em uma única passada em lote
dataset = dataset.map(formatting_prompts_func, batched=True)

# Sanity check — inspect one formatted prompt to confirm special tokens are present
print(dataset["train"][0]["text"])


## 7. Load gemma4-E2B-it in 4-bit for QLoRA

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


In [ ]:
from transformers import (
    AutoModelForImageTextToText,  
    BitsAndBytesConfig,
)


In [ ]:
# Quantisation config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   
    bnb_4bit_quant_type="nf4",        
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model 
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto", 
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

# Report memory usage
print("First parameter device:", next(model.parameters()).device)
print("CUDA allocated GB:", torch.cuda.memory_allocated() / 1e9)
print("CUDA reserved  GB:", torch.cuda.memory_reserved()  / 1e9)
print("Memory footprint bytes:", model.get_memory_footprint())


## 8. QLoRA configuration

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,                          # rank das matrizes do adaptador
    lora_alpha=32,                 # scaling: effective update magnitude = alpha/r * ΔW
    lora_dropout=0.0,              # no dropout 
    bias="none",                   # don't train bias terms
    target_modules="all-linear",   # aplica LoRA a todas as camadas lineares do modelo
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,      
)

peft_config


## 9. Configuração de treinamento

Decisões principais:

- **`max_length=768`** — cobre os prompts mais longos do corpus sem
  padding excessivo nos exemplos mais curtos.
- **`gradient_accumulation_steps=8`** with `per_device_train_batch_size=1`
  fornece um tamanho efetivo de batch de 8, um compromisso razoável
  entre estabilidade e VRAM em uma única GPU.
- **`bf16=True`** — o cálculo em bfloat16 corresponde ao dtype de armazenamento do modelo e
  evita picos de loss que podem ocorrer com float16 em modelos grandes.
- **Cosine LR schedule with 10 % warmup** — standard practice for
  instruction fine-tuning; prevents early destructive updates.
- **`load_best_model_at_end=True`** on `eval_loss` — automatically
  salva checkpoint da melhor época em vez da última.


In [ ]:
from trl import SFTConfig

args = SFTConfig(
    output_dir="port-a-prof_training",   # intermediate checkpoints go here

    # Data 
    dataset_text_field="text",           
    max_length=768,                      

    # Training schedule 
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      

    # Optimiser
    optim="adamw_torch_fused",           
    learning_rate=1e-4,
    max_grad_norm=1.0,                   
    warmup_ratio=0.1,                    
    lr_scheduler_type="cosine",

    # Precision
    bf16=True,
    fp16=False,

    # Checkpointing & evaluation
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   
    save_total_limit=2,                

    # Reporting
    report_to="tensorboard",
    push_to_hub=False,

    # Tokenisation
    # O template de chat já inclui todos os tokens especiais necessários, então nós
    # must not add them again here.
    dataset_kwargs={
        "add_special_tokens":  False,
        "append_concat_token": False,
    },
)


## 10. Create trainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,           # injeta adaptadores LoRA no modelo
    processing_class=processor.tokenizer,
)

trainer


## 11. Train

In [ ]:
trainer.train()

# Salva o melhor checkpoint (pesos do adaptador LoRA + estado do trainer)
trainer.save_model()


## 12. Free memory

Exclui o objeto trainer e limpa o cache CUDA antes de executar inferência,
para que a GPU tenha memória livre suficiente para carregar o modelo adaptado para geração.


In [ ]:
del trainer
torch.cuda.empty_cache()


## 13. Inference test


In [ ]:
# Pick a validation example 
sample = dataset["validation"][1]

# Usa apenas o turno do usuário — o modelo deve gerar o turno do assistente
messages = sample["messages"][:1]
print(messages)

# Aplica o template de chat com add_generation_prompt=True para que o modelo saiba
# começar a gerar imediatamente após o turno do usuário
input_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=== INPUT PROMPT ===\n")
print(input_text)

# Tokenise and move to GPU
inputs = processor(text=input_text, return_tensors="pt")
inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

# Gera — sem necessidade de rastrear gradientes durante a inferência
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=1000)

# Remove os tokens do prompt para decodificar apenas a resposta gerada
prompt_len  = inputs["input_ids"].shape[1]
gen_tokens  = out[0][prompt_len:]
gen         = processor.decode(gen_tokens, skip_special_tokens=True)

print("\n=== MODEL OUTPUT ===\n")
print(gen.strip())

print("\n=== REFERENCE ===\n")
print(sample["messages"][1]["content"])


## 14. Merge adapter and save final model

In [ ]:
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch

# 1. Reload base model in full bfloat16 — merging requires unquantised weights
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

# 2. Anexa o adaptador LoRA treinado (output_dir do SFTConfig)
peft_model = PeftModel.from_pretrained(base_model, args.output_dir)

# 3. Funde os pesos do adaptador no modelo base e descarta a estrutura do adaptador
merged_model = peft_model.merge_and_unload()

# 4. Salva
merged_model.save_pretrained(
    "./port_a_prof_finetuned",
    safe_serialization=True,
    max_shard_size="2GB",
)

# 5. Salva o processor junto com o modelo para que tudo necessário à inferência
#    lives in one directory
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
processor.save_pretrained("./port_a_prof_finetuned")
